WhatNet  –  Persian

In [ ]:
#  Persian digits (۰–۹) are written in the Perso-Arabic script right-to-left.
#  They share visual roots with Arabic-Indic numerals but differ in glyphs for
#  ۴ (4), ۵ (5), and ۶ (6).  Characters are isolated (non-cursive digits),
#  rounded, and often feature interior loops and ascenders.

#    • num_classes   : 10   (same count: Persian digits ۰–۹)
#    • image_size    : 32   (HODA images are 32×32 greyscale)
#    • Dataset format: .npz arrays → now supports FOUR layouts, auto-detected:
#         1. CDB  – Train 60000.cdb + Test 20000.cdb  via HodaDatasetReader
#         2. MAT  – Data_hoda_full.mat                via scipy.io.loadmat
#         3. CSV  – train.csv + test.csv              (flat pixel arrays)
#         4. DIR  – Train/<digit>/*.png + Test/...    (folder-per-class)
#    • Dataset loader: npz → quad-mode auto-detect
#    • Scaffold branch: 1×3 (Kannada cross-hair) → 3×3 (isotropic Persian
#                       rounded strokes with interior loops)
#    • Augmentation  : random_flip_left_right REMOVED – Persian digit glyphs
#                       are NOT symmetric; ۶ would be confused with a mirrored
#                       form.  Only brightness/contrast + pad-crop applied.

#  Dataset
#  HODA – Handwritten Farsi Digit Dataset  (Khosravi & Kabir, 2007)
#    • ~60 000 train + ~20 000 test 32×32 greyscale images of Persian digits
#
#  Expected directory layout (CFG["data_dir"]):
#
#    Layout 1 – raw CDB (preferred, highest fidelity):
#      <data_dir>/DigitDB/Train 60000.cdb
#      <data_dir>/DigitDB/Test 20000.cdb
#      <data_dir>/HodaDatasetReader.py        ← must be importable
#
#    Layout 2 – MAT export (fallback if CDB reader absent):
#      <data_dir>/archive/Data_hoda_full.mat
#      Expected keys: 'X' (N×1024 uint8) and 'Y' (N×1 or N, int)
#      OR the keys your specific export uses (script will print actual keys).
#
#    Layout 3 – CSV export:
#      <data_dir>/train.csv    col 0 = label, cols 1-1024 = pixels
#      <data_dir>/test.csv
#
#    Layout 4 – folder-per-class PNG tree:
#      <data_dir>/Train/0/*.png … /Train/9/*.png
#      <data_dir>/Test/0/*.png  … /Test/9/*.png
#
#  The script probes in the order 1 → 2 → 3 → 4 and uses the first match.
#  Kaggle notebook path example:
#      CFG["data_dir"] = "/kaggle/input/hoda-farsi-digit-dataset"

In [ ]:
# importing necessary dependencies
import os, sys, time, random, json
import numpy as np
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from PIL import Image as PilImage
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

#  0. REPRODUCIBILITY
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

#  1. CONFIGURATION
CFG = {
    # Persian digits ۰ ۱ ۲ ۳ ۴ ۵ ۶ ۷ ۸ ۹
    "num_classes":      10,
    "image_size":       32,
    "batch_size":       64,
    "epochs":           100,
    "lr":               5e-4,
    "weight_decay":     1e-4,
    "label_smoothing":  0.1,
    "val_split":        0.1,

    #   root of the HODA dataset directory.
    #   Set this to wherever you extracted the Kaggle dataset.
    #   Kaggle notebook: "/kaggle/input/hoda-farsi-digit-dataset"
    "data_dir":    "/kaggle/input/datasets/invalizare/hoda-mat",

    "results_dir": "./results",

    #  CDB-specific
    # File names inside <data_dir>/DigitDB/  (change if your zip differs)
    "cdb_train":   "Train 60000.cdb",
    "cdb_test":    "Test 20000.cdb",

    # MAT-specific
    # scipy.io.loadmat key names for images and labels.
    # The script will print the actual keys found if these don't match.
    "mat_file":    "Data_hoda_full.mat",   # relative to <data_dir>/archive/
    "mat_x_key":   "X",                   # shape (N, 1024) uint8
    "mat_y_key":   "Y",                   # shape (N,) or (N,1) int
    # Fraction of MAT data used as test set (no separate test split in .mat)
    "mat_test_frac": 0.25,
}
os.makedirs(CFG["results_dir"], exist_ok=True)

NUM_CLASSES = CFG["num_classes"]
IMG         = CFG["image_size"]
BS          = CFG["batch_size"]
AUTOTUNE    = tf.data.AUTOTUNE

#  2. DATASET PIPELINE
#     Four layouts probed in priority order: CDB → MAT → CSV → DIR
if not os.path.exists(CFG["data_dir"]):
    raise FileNotFoundError(
        f"[ERROR] Dataset directory not found: {CFG['data_dir']}\n"
        "Download HODA from Kaggle:\n"
        "  https://www.kaggle.com/datasets/mostafij/hoda-farsi-digit-dataset\n"
        "Supported layouts inside CFG['data_dir']:\n"
        "  1) DigitDB/Train 60000.cdb + DigitDB/Test 20000.cdb + HodaDatasetReader.py\n"
        "  2) archive/Data_hoda_full.mat\n"
        "  3) train.csv + test.csv\n"
        "  4) Train/<digit>/*.png + Test/<digit>/*.png"
    )

# Shared helpers
def _split_train_val(x, y):
    """Shuffle and split off a validation fold from training arrays."""
    n_total = len(x)
    n_val   = max(1, int(n_total * CFG["val_split"]))
    n_train = n_total - n_val
    rng     = np.random.default_rng(SEED)
    perm    = rng.permutation(n_total)
    x, y    = x[perm], y[perm]
    return x[:n_train], y[:n_train], x[n_train:], y[n_train:], n_train, n_val

def _arrays_to_ds(x, y, shuffle=False, augment_flag=False, one_hot=True):
    """
    Build a tf.data pipeline from numpy arrays.
    x: (N, H, W) uint8 or float32 – will be expanded to (N, H, W, 1).
    y: (N,) int32.
    """
    if x.ndim == 3:
        x = x[..., np.newaxis]

    def normalise(img, lbl):
        img = tf.cast(img, tf.float32) / 127.5 - 1.0
        return img, lbl

    def augment_fn(img, lbl):
        img = tf.image.random_brightness(img, 0.2)
        img = tf.image.random_contrast(img, 0.8, 1.2)
        img = tf.pad(img, [[2, 2], [2, 2], [0, 0]], constant_values=-1.0)
        img = tf.image.random_crop(img, [IMG, IMG, 1])
        return img, lbl

    def to_onehot(img, lbl):
        return img, tf.one_hot(lbl, NUM_CLASSES)

    ds = tf.data.Dataset.from_tensor_slices((x, y.astype(np.int32)))
    ds = ds.map(normalise, num_parallel_calls=AUTOTUNE)
    if augment_flag:
        ds = ds.map(augment_fn, num_parallel_calls=AUTOTUNE)
    if one_hot:
        ds = ds.map(to_onehot, num_parallel_calls=AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(8192, seed=SEED)
    ds = ds.batch(BS).prefetch(AUTOTUNE)
    return ds

# Mode 1: CDB via HodaDatasetReader
_digitdb_dir   = os.path.join(CFG["data_dir"], "DigitDB")
_cdb_train     = os.path.join(_digitdb_dir, CFG["cdb_train"])
_cdb_test      = os.path.join(_digitdb_dir, CFG["cdb_test"])
_reader_script = os.path.join(CFG["data_dir"], "HodaDatasetReader.py")

USE_CDB = (
    os.path.exists(_cdb_train)
    and os.path.exists(_cdb_test)
    and os.path.exists(_reader_script)
)

# Mode 2: MAT via scipy
_mat_path = os.path.join(CFG["data_dir"], "archive", CFG["mat_file"])
USE_MAT   = (not USE_CDB) and os.path.exists(_mat_path)

# Mode 3: CSV
_train_csv = os.path.join(CFG["data_dir"], "train.csv")
_test_csv  = os.path.join(CFG["data_dir"], "test.csv")
USE_CSV    = (not USE_CDB) and (not USE_MAT) and (
    os.path.exists(_train_csv) and os.path.exists(_test_csv)
)

# Mode 4: Folder-per-class PNG tree
_train_dir = os.path.join(CFG["data_dir"], "Train")
_test_dir  = os.path.join(CFG["data_dir"], "Test")
USE_DIR    = (not USE_CDB) and (not USE_MAT) and (not USE_CSV) and (
    os.path.exists(_train_dir) and os.path.exists(_test_dir)
)

if USE_CDB:
    # Layout 1: raw CDB files
    print("[INFO] HODA layout detected: CDB mode (HodaDatasetReader)")

    # Make HodaDatasetReader importable from the dataset directory
    if CFG["data_dir"] not in sys.path:
        sys.path.insert(0, CFG["data_dir"])
    from HodaDatasetReader import read_hoda_cdb   # noqa: E402

    print(f"[INFO] Reading train CDB: {_cdb_train}")
    train_images_raw, train_labels_raw = read_hoda_cdb(_cdb_train)
    print(f"[INFO] Reading test  CDB: {_cdb_test}")
    test_images_raw,  test_labels_raw  = read_hoda_cdb(_cdb_test)

    # read_hoda_cdb returns lists of 2-D numpy arrays (H×W, uint8).
    # Resize all images to IMG×IMG and stack into a single array.
    def _resize_to_array(images_list):
        out = []
        for img in images_list:
            if img.shape != (IMG, IMG):
                pil = PilImage.fromarray(img).resize((IMG, IMG), PilImage.BILINEAR)
                img = np.array(pil, dtype=np.uint8)
            out.append(img)
        return np.stack(out, axis=0)          # (N, IMG, IMG)

    x_train_all = _resize_to_array(train_images_raw)
    y_train_all = np.array(train_labels_raw, dtype=np.int32)
    x_test      = _resize_to_array(test_images_raw)
    y_test      = np.array(test_labels_raw,  dtype=np.int32)

    print(f"[INFO] Classes in dataset : {np.unique(y_train_all).tolist()}")
    x_train, y_train, x_val, y_val, n_train, n_val = _split_train_val(
        x_train_all, y_train_all
    )
    print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: {len(x_test):,}")

    train_ds      = _arrays_to_ds(x_train, y_train, shuffle=True,  augment_flag=True,  one_hot=True)
    val_ds        = _arrays_to_ds(x_val,   y_val,   shuffle=False, augment_flag=False, one_hot=True)
    test_ds       = _arrays_to_ds(x_test,  y_test,  shuffle=False, augment_flag=False, one_hot=False)
    test_ds_oh    = _arrays_to_ds(x_test,  y_test,  shuffle=False, augment_flag=False, one_hot=True)
    steps_per_epoch = -(-n_train // BS)

elif USE_MAT:
    # Layout 2: .mat file via scipy
    print(f"[INFO] HODA layout detected: MAT mode ({_mat_path})")
    try:
        import scipy.io as sio
    except ImportError:
        raise ImportError(
            "[ERROR] scipy is required for MAT loading.\n"
            "  Install it with:  pip install scipy"
        )

    mat = sio.loadmat(_mat_path)
    # Print the actual variable names so the user can adjust CFG if needed
    data_keys = [k for k in mat.keys() if not k.startswith("__")]
    print(f"[INFO] Keys found in .mat file: {data_keys}")

    x_key = CFG["mat_x_key"]
    y_key = CFG["mat_y_key"]
    if x_key not in mat or y_key not in mat:
        raise KeyError(
            f"[ERROR] Expected keys '{x_key}' and '{y_key}' in MAT file.\n"
            f"  Actual keys: {data_keys}\n"
            f"  Update CFG['mat_x_key'] and CFG['mat_y_key'] to match."
        )

    x_all = mat[x_key].astype(np.float32)   # (N, 1024) expected
    y_all = mat[y_key].astype(np.int32).ravel()

    # Reshape flat pixel vectors → 2-D images
    if x_all.ndim == 2 and x_all.shape[1] == IMG * IMG:
        x_all = x_all.reshape(-1, IMG, IMG)
    elif x_all.ndim == 3:
        pass  # already (N, H, W)
    else:
        raise ValueError(
            f"[ERROR] Unexpected image array shape from MAT: {x_all.shape}\n"
            f"  Expected (N, {IMG*IMG}) or (N, {IMG}, {IMG})."
        )

    print(f"[INFO] Total samples in .mat: {len(x_all):,}")
    print(f"[INFO] Classes found        : {np.unique(y_all).tolist()}")

    # The .mat file contains a single pool; carve out test + val manually
    n_total_mat = len(x_all)
    n_test  = max(1, int(n_total_mat * CFG["mat_test_frac"]))
    rng     = np.random.default_rng(SEED)
    perm    = rng.permutation(n_total_mat)
    x_all, y_all = x_all[perm], y_all[perm]

    x_test, y_test     = x_all[:n_test],  y_all[:n_test]
    x_trainval         = x_all[n_test:]
    y_trainval         = y_all[n_test:]

    n_tv  = len(x_trainval)
    n_val = max(1, int(n_tv * CFG["val_split"]))
    n_train = n_tv - n_val
    x_train, y_train = x_trainval[:n_train], y_trainval[:n_train]
    x_val,   y_val   = x_trainval[n_train:], y_trainval[n_train:]

    print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: {len(x_test):,}")

    train_ds      = _arrays_to_ds(x_train, y_train, shuffle=True,  augment_flag=True,  one_hot=True)
    val_ds        = _arrays_to_ds(x_val,   y_val,   shuffle=False, augment_flag=False, one_hot=True)
    test_ds       = _arrays_to_ds(x_test,  y_test,  shuffle=False, augment_flag=False, one_hot=False)
    test_ds_oh    = _arrays_to_ds(x_test,  y_test,  shuffle=False, augment_flag=False, one_hot=True)
    steps_per_epoch = -(-n_train // BS)

elif USE_CSV:
    # Layout 3: CSV arrays
    print("[INFO] HODA layout detected: CSV mode")

    def _load_csv_split(split: str):
        csv_path = os.path.join(CFG["data_dir"], f"{split}.csv")
        print(f"[INFO] Loading {split} from CSV: {csv_path}")
        raw    = np.loadtxt(csv_path, delimiter=",", skiprows=1, dtype=np.float32)
        labels = raw[:, 0].astype(np.int32)
        images = raw[:, 1:].reshape(-1, IMG, IMG).astype(np.float32)
        return images, labels

    x_train_all, y_train_all = _load_csv_split("train")
    x_test,      y_test      = _load_csv_split("test")

    print(f"[INFO] Classes in dataset : {np.unique(y_train_all).tolist()}")
    x_train, y_train, x_val, y_val, n_train, n_val = _split_train_val(
        x_train_all, y_train_all
    )
    print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: {len(x_test):,}")

    train_ds      = _arrays_to_ds(x_train, y_train, shuffle=True,  augment_flag=True,  one_hot=True)
    val_ds        = _arrays_to_ds(x_val,   y_val,   shuffle=False, augment_flag=False, one_hot=True)
    test_ds       = _arrays_to_ds(x_test,  y_test,  shuffle=False, augment_flag=False, one_hot=False)
    test_ds_oh    = _arrays_to_ds(x_test,  y_test,  shuffle=False, augment_flag=False, one_hot=True)
    steps_per_epoch = -(-n_train // BS)

elif USE_DIR:
    # Layout 4: folder-per-class PNG tree
    print("[INFO] HODA layout detected: folder-per-class mode")

    def _collect_paths_and_labels(root: str):
        class_names = sorted(
            d for d in os.listdir(root)
            if os.path.isdir(os.path.join(root, d))
        )
        paths, labels = [], []
        for idx, cls in enumerate(class_names):
            folder = os.path.join(root, cls)
            for fname in sorted(os.listdir(folder)):
                if fname.lower().endswith((".png", ".jpg", ".jpeg", ".bmp")):
                    paths.append(os.path.join(folder, fname))
                    labels.append(idx)
        return paths, labels, class_names

    def _pil_load_fn(path_tensor, label_tensor):
        def _load(path_bytes):
            path = path_bytes.numpy().decode("utf-8")
            img  = PilImage.open(path).convert("L")
            img  = img.resize((IMG, IMG), PilImage.BILINEAR)
            arr  = np.array(img, dtype=np.float32)[:, :, np.newaxis]
            return arr
        img = tf.py_function(_load, [path_tensor], tf.float32)
        img.set_shape((IMG, IMG, 1))
        return img, label_tensor

    def _make_ds_from_dir(root: str, shuffle: bool = False) -> tf.data.Dataset:
        paths, labels, _ = _collect_paths_and_labels(root)
        ds = tf.data.Dataset.from_tensor_slices(
            (tf.constant(paths), tf.constant(labels, dtype=tf.int32))
        )
        if shuffle:
            ds = ds.shuffle(len(paths), seed=SEED)
        ds = ds.map(_pil_load_fn, num_parallel_calls=AUTOTUNE)
        return ds

    _, _, _cls = _collect_paths_and_labels(_train_dir)
    print(f"[INFO] Classes found on disk: {len(_cls)}  |  CFG num_classes: {NUM_CLASSES}")

    train_full  = _make_ds_from_dir(_train_dir, shuffle=True)
    test_ds_raw = _make_ds_from_dir(_test_dir,  shuffle=False)

    total        = sum(1 for _ in train_full)
    n_val        = max(1, int(total * CFG["val_split"]))
    n_train      = total - n_val
    train_ds_raw = train_full.take(n_train)
    val_ds_raw   = train_full.skip(n_train)
    print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: (batched from disk)")

    def normalise(img, lbl):
        img = tf.cast(img, tf.float32) / 127.5 - 1.0
        return img, lbl

    def augment_fn(img, lbl):
        img = tf.image.random_brightness(img, 0.2)
        img = tf.image.random_contrast(img, 0.8, 1.2)
        img = tf.pad(img, [[2, 2], [2, 2], [0, 0]], constant_values=-1.0)
        img = tf.image.random_crop(img, [IMG, IMG, 1])
        return img, lbl

    def to_onehot(img, lbl):
        return img, tf.one_hot(lbl, NUM_CLASSES)

    train_ds = (
        train_ds_raw
        .map(normalise,  num_parallel_calls=AUTOTUNE)
        .map(augment_fn, num_parallel_calls=AUTOTUNE)
        .map(to_onehot,  num_parallel_calls=AUTOTUNE)
        .shuffle(8192, seed=SEED)
        .batch(BS).prefetch(AUTOTUNE)
    )
    val_ds = (
        val_ds_raw
        .map(normalise,  num_parallel_calls=AUTOTUNE)
        .map(to_onehot,  num_parallel_calls=AUTOTUNE)
        .batch(BS).prefetch(AUTOTUNE)
    )
    test_ds = (
        test_ds_raw
        .map(normalise,  num_parallel_calls=AUTOTUNE)
        .batch(BS).prefetch(AUTOTUNE)
    )
    test_ds_oh = (
        test_ds_raw
        .map(normalise,  num_parallel_calls=AUTOTUNE)
        .map(to_onehot,  num_parallel_calls=AUTOTUNE)
        .batch(BS).prefetch(AUTOTUNE)
    )
    steps_per_epoch = sum(1 for _ in train_ds)

else:
    raise FileNotFoundError(
        "[ERROR] Could not find HODA data in any supported layout.\n"
        f"  data_dir = {CFG['data_dir']}\n\n"
        "  Layout 1 (CDB):  DigitDB/Train 60000.cdb  +  DigitDB/Test 20000.cdb\n"
        "                   HodaDatasetReader.py  (in data_dir root)\n"
        "  Layout 2 (MAT):  archive/Data_hoda_full.mat\n"
        "  Layout 3 (CSV):  train.csv  +  test.csv\n"
        "  Layout 4 (DIR):  Train/<digit>/*.png  +  Test/<digit>/*.png\n\n"
        "  If your .cdb reader script has a different name, set:\n"
        "      CFG['cdb_train'] / CFG['cdb_test'] / and place\n"
        "      HodaDatasetReader.py in CFG['data_dir']."
    )

#  3. DISPLAY UTILITIES  (unchanged)
_COL = {
    "reset":  "\033[0m",  "bold":   "\033[1m",  "cyan":   "\033[96m",
    "yellow": "\033[93m", "green":  "\033[92m",  "red":    "\033[91m",
    "grey":   "\033[90m", "white":  "\033[97m",  "blue":   "\033[94m",
}

def _c(text, *codes):
    prefix = "".join(_COL.get(c, "") for c in codes)
    return f"{prefix}{text}{_COL['reset']}"

def print_model_summary(model: Model) -> None:
    W             = 62
    trainable     = model.count_params()
    non_trainable = sum(tf.size(w).numpy() for w in model.non_trainable_weights)
    total         = trainable + non_trainable
    title         = f"  {model.name}  –  Parameter Summary"
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan')}")
    print(_c(f"║{title:<{W}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*18}╦{'═'*23}╦{'═'*18}╣", "cyan"))
    print(_c(f"║  {'Layer':<16}║  {'Type':<21}║  {'Params':>15}  ║", "cyan", "bold"))
    print(_c(f"╠{'═'*18}╬{'═'*23}╬{'═'*18}╣", "cyan"))
    for lyr in [l for l in model.layers if l.count_params() > 0][:20]:
        n_p = lyr.count_params()
        print(f"║  {lyr.name[:14]:<16}║  {type(lyr).__name__[:21]:<21}║  {n_p:>15,}  ║")
    if len([l for l in model.layers if l.count_params() > 0]) > 20:
        print(f"║  {'… (truncated)':<16}║  {'':21}║  {'':>15}  ║")
    print(_c(f"╠{'═'*18}╩{'═'*23}╩{'═'*18}╣", "cyan"))
    print(_c(f"║  {'Trainable params':<38}: {trainable:>18,}  ║", "green"))
    print(_c(f"║  {'Non-trainable params':<38}: {non_trainable:>18,}  ║", "grey"))
    print(_c(f"║  {'Total params':<38}: {total:>18,}  ║", "bold", "white"))
    print(_c(f"╚{'═'*W}╝", "cyan"))

class EpochProgressCallback(keras.callbacks.Callback):
    BAR_WIDTH = 20
    def __init__(self, total_epochs: int, model_name: str):
        super().__init__()
        self.total_epochs = total_epochs
        self.model_name   = model_name
        self._epoch_start = 0.0

    def on_epoch_begin(self, epoch, logs=None):
        self._epoch_start = time.time()

    def on_epoch_end(self, epoch, logs=None):
        logs    = logs or {}
        elapsed = time.time() - self._epoch_start
        ep_num  = epoch + 1
        pct     = ep_num / self.total_epochs
        filled  = int(self.BAR_WIDTH * pct)
        bar     = "█" * filled + "░" * (self.BAR_WIDTH - filled)
        loss    = logs.get("loss",         float("nan"))
        acc     = logs.get("accuracy",     float("nan")) * 100
        val_acc = logs.get("val_accuracy", float("nan")) * 100
        val_los = logs.get("val_loss",     float("nan"))
        try:
            lr_val = float(keras.backend.get_value(self.model.optimizer.learning_rate))
            lr_str = f"lr={lr_val:.2e}"
        except Exception:
            lr_str = ""
        print(
            f"  {_c(f'Epoch {ep_num:>3}/{self.total_epochs}', 'grey')}  "
            f"{_c(bar, 'cyan')} {_c(f'{pct*100:>5.1f}%', 'yellow')}  "
            f"{_c(f'loss={loss:.4f}', 'white')}  {_c(f'acc={acc:.2f}%', 'green')}  "
            f"{_c(f'val_loss={val_los:.4f}', 'white')}  "
            f"{_c(f'val_acc={val_acc:.2f}%', 'yellow' if val_acc < acc else 'green')}  "
            f"{_c(lr_str, 'blue')}  {_c(f'[{elapsed:.1f}s]', 'grey')}"
        )

def print_comparison_table(results: dict) -> None:
    W         = 70
    best_name = max(results, key=lambda k: results[k]["test_acc"])
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan', 'bold')}")
    print(_c(f"║  {'FINAL TEST-SET COMPARISON':<{W-2}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*24}╦{'═'*12}╦{'═'*12}╦{'═'*12}╦{'═'*6}╣", "cyan"))
    print(_c(f"║  {'Model':<22}║{'Params':>11} ║{'Test Acc':>11} ║{'Macro F1':>11} ║{'Loss':>5} ║",
             "cyan", "bold"))
    print(_c(f"╠{'═'*24}╬{'═'*12}╬{'═'*12}╬{'═'*12}╬{'═'*6}╣", "cyan"))
    for name, r in results.items():
        is_best = (name == best_name)
        star    = "★" if is_best else " "
        row = (f"║{star} {name:<22}║{r['params']:>10,} ║"
               f"{r['test_acc']:>10.2f}%║{r['macro_f1']:>10.2f}%║{r['test_loss']:>5.3f} ║")
        print(_c(row, "green", "bold") if is_best else row)
    print(_c(f"╚{'═'*24}╩{'═'*12}╩{'═'*12}╩{'═'*12}╩{'═'*6}╝", "cyan"))
    print(_c(f"\n  ★  Winner: {best_name}  ({results[best_name]['test_acc']:.2f}% test accuracy)\n",
             "green", "bold"))

#  4. BUILDING BLOCKS
def gelu(x):
    return tf.nn.gelu(x)

def residual_block(x, channels: int):
    residual = x
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(gelu)(x)
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([x, residual])
    x = layers.Activation(gelu)(x)
    return x

def dense_res_block(x, in_channels: int, out_channels: int):
    if in_channels != out_channels:
        skip = layers.Conv2D(out_channels, 1, use_bias=False)(x)
        skip = layers.BatchNormalization()(skip)
        x_in = layers.Activation(gelu)(skip)
    else:
        x_in = x
    r1  = residual_block(x_in, out_channels)
    r2  = residual_block(r1,   out_channels)
    r3  = residual_block(r2,   out_channels)
    cat = layers.Concatenate()([r1, r2, r3])
    out = layers.Conv2D(out_channels, 1, use_bias=False)(cat)
    out = layers.BatchNormalization()(out)
    out = layers.Activation(gelu)(out)
    out = layers.DepthwiseConv2D(3, strides=2, padding="same", use_bias=False)(out)
    out = layers.Conv2D(out_channels, 1, use_bias=False)(out)
    out = layers.BatchNormalization()(out)
    out = layers.Activation(gelu)(out)
    return out

def channel_attention(x, channels: int, reduction: int = 8):
    gap  = layers.GlobalAveragePooling2D(keepdims=True)(x)
    gap  = layers.Reshape((channels,))(gap)
    attn = layers.Dense(channels // reduction, activation="relu")(gap)
    attn = layers.Dense(channels, activation="sigmoid")(attn)
    attn = layers.Reshape((1, 1, channels))(attn)
    return layers.Multiply()([x, attn])

def adaptive_filter_capsule(x, num_classes: int, capsule_dim: int = 16):
    h        = layers.Dense(256, activation=gelu)(x)
    h        = layers.Dense(num_classes * capsule_dim)(h)
    h        = layers.Reshape((num_classes, capsule_dim))(h)
    x_exp    = layers.RepeatVector(num_classes)(x)
    x_sliced = layers.Lambda(lambda t: t[:, :, :capsule_dim])(x_exp)
    caps     = layers.Multiply()([x_sliced, h])
    caps     = layers.Lambda(lambda t: tf.reduce_sum(t, axis=-1))(caps)
    caps     = layers.BatchNormalization()(caps)
    return caps

#  5. MODEL DEFINITION
def build_whatnet_persian(num_classes: int = 10,
    drop_path_rate: float = 0.05,
    dropout_rate: float = 0.3,
    weight_decay: float = 1e-4,
    head_units: int = 256,
    override_tier: int = None,
    image_size: int = 32) -> Model:
    """
    WhatNet-Persian: WhatNet adapted for HODA Persian digit recognition.

    Architecture overview
    Stem (dual-path):
      -> Standard 3×3 conv path (glyph body / overall shape)
      -> Loop-aware scaffold: 3×3 isotropic conv – captures the full rounded
        stroke body of Persian digits, including interior loops
      -> Concatenated and refined with SE channel attention

    Encoder (3 stages, each halving spatial dims):
      enc1: 64→64    (16×16 at 32-px input)
      enc2: 64→128   ( 8× 8)
      enc3: 128→256  ( 4× 4)
      Weighted scaffold residuals injected at each stage.

    Decoder head:
      Multi-scale GAP
      AFC
      Classification Head
      Gated fusion → final MLP + layer norm → logits (10 classes)
    """
    K   = num_classes
    inp = Input(shape=(image_size, image_size, 1), name="input")

    # Stem
    t        = layers.Conv2D(32, 3, padding="same", use_bias=False)(inp)
    t        = layers.BatchNormalization()(t)
    t        = layers.Activation(gelu)(t)

    s        = layers.Conv2D(32, 3, padding="same", use_bias=False)(inp)
    s        = layers.BatchNormalization()(s)
    scaffold = layers.Activation(gelu)(s)

    stem = layers.Concatenate()([t, scaffold])
    stem = channel_attention(stem, 64)
    stem = layers.Conv2D(64, 1, use_bias=False)(stem)
    stem = layers.BatchNormalization()(stem)
    stem = layers.Activation(gelu)(stem)

    # Encoder
    enc1 = dense_res_block(stem, 64, 64)
    sc1  = layers.AveragePooling2D(2)(layers.Conv2D(64,  1, use_bias=False)(scaffold))
    enc1 = layers.Add()([enc1, layers.Lambda(lambda t: t * 0.1)(sc1)])

    enc2 = dense_res_block(enc1, 64, 128)
    sc2  = layers.AveragePooling2D(4)(layers.Conv2D(128, 1, use_bias=False)(scaffold))
    enc2 = layers.Add()([enc2, layers.Lambda(lambda t: t * 0.1)(sc2)])

    enc3 = dense_res_block(enc2, 128, 256)
    sc3  = layers.AveragePooling2D(8)(layers.Conv2D(256, 1, use_bias=False)(scaffold))
    enc3 = layers.Add()([enc3, layers.Lambda(lambda t: t * 0.1)(sc3)])

    # Decoder head
    gap1 = layers.GlobalAveragePooling2D(name="gap1")(enc1)
    gap2 = layers.GlobalAveragePooling2D(name="gap2")(enc2)
    gap3 = layers.GlobalAveragePooling2D(name="gap3")(enc3)
    fused_gap = layers.Concatenate(name="multiscale_fused")([gap1, gap2, gap3])

    afc_out = adaptive_filter_capsule(fused_gap, num_classes)   # (B, K)

    # Classification head
    # Dense projection of the raw GAP features (residual path alongside AFC)
    x = layers.Dense(head_units, use_bias=False, name="head_dense")(fused_gap)
    x = layers.LayerNormalization(name="head_ln")(x)
    x = layers.Activation("gelu", name="head_act")(x)
    x = layers.Dense(num_classes, name="head_logits")(x)

    # Gated fusion: AFC scores + dense-head logits
    # A learned scalar gate (per-sample softmax over 2 weights) blends the
    # AFC capsule scores with the plain dense logits.  This lets the model
    # learn how much to trust the capsule routing vs. the direct projection.
    combined = layers.Concatenate(name="gate_input")([x, afc_out])
    gate     = layers.Dense(2, activation="softmax", name="gate")(combined)  # (B, 2)

    # gate[:,0] weights the dense head; gate[:,1] weights the AFC output
    x_scaled   = layers.Lambda(
        lambda t: t[0] * t[1][:, 0:1], name="gate_dense")([x,gate])
    afc_scaled = layers.Lambda(
        lambda t: t[0] * t[1][:, 1:2], name="gate_afc"  )([afc_out,gate])

    outputs = layers.Add(name="logits")([x_scaled, afc_scaled])


    return Model(inputs=inp, outputs=outputs, name="WhatNet-Persian")

MODELS_TF = {
    "WhatNet-Persian": lambda: build_whatnet_persian(NUM_CLASSES, IMG),
}

#  6. LR SCHEDULE
class CosineAnnealing(keras.optimizers.schedules.LearningRateSchedule):
    """Cosine-annealing without restarts: lr(t) = max(base·½·(1+cos(π·t/T)), 1e-6)."""
    def __init__(self, base: float, steps: int):
        self.base  = base
        self.steps = tf.cast(steps, tf.float32)

    def __call__(self, step):
        step   = tf.cast(step, tf.float32)
        cosine = 0.5 * (1.0 + tf.cos(np.pi * step / self.steps))
        return tf.maximum(self.base * cosine, 1e-6)

    def get_config(self):
        return {"base": self.base, "steps": int(self.steps)}

#  7. TRAINING & EVALUATION HELPERS
def compile_model(model: Model, steps_total: int) -> Model:
    lr_sch = CosineAnnealing(CFG["lr"], steps_total)
    model.compile(
        optimizer=keras.optimizers.AdamW(
            learning_rate=lr_sch,
            weight_decay=CFG["weight_decay"],
        ),
        loss=keras.losses.CategoricalCrossentropy(
            from_logits=True,
            label_smoothing=CFG["label_smoothing"],
        ),
        metrics=["accuracy"],
        jit_compile=True,
    )
    return model

def compute_macro_f1(model: Model, dataset) -> float:
    """Macro-averaged F1 over all NUM_CLASSES classes (returns %)."""
    tp = np.zeros(NUM_CLASSES)
    fp = np.zeros(NUM_CLASSES)
    fn = np.zeros(NUM_CLASSES)
    for images, labels in dataset:
        preds = tf.argmax(model(images, training=False), axis=1).numpy()
        lbls  = labels.numpy()
        for c in range(NUM_CLASSES):
            tp[c] += np.sum((preds == c) & (lbls == c))
            fp[c] += np.sum((preds == c) & (lbls != c))
            fn[c] += np.sum((preds != c) & (lbls == c))
    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    f1   = 2 * prec * rec / (prec + rec + 1e-8)
    return float(f1.mean() * 100.0)

#  8. TRAIN + EVALUATE
trained_models = {}
all_histories  = {}
total_steps    = steps_per_epoch * CFG["epochs"]

print(_c(f"\n{'═'*70}", "cyan"))
print(_c(f"  Starting benchmark: {len(MODELS_TF)} model(s) × {CFG['epochs']} epochs"
         f"  [HODA – Persian]", "bold"))
print(_c(f"{'═'*70}\n", "cyan"))

for name, model_fn in MODELS_TF.items():
    model = model_fn()
    model = compile_model(model, total_steps)
    print_model_summary(model)

    ckpt_path = os.path.join(CFG["results_dir"], f"{name}_best.keras")
    callbacks = [
        ModelCheckpoint(ckpt_path, monitor="val_accuracy", save_best_only=True, verbose=0),
        EarlyStopping(monitor="val_accuracy", patience=15, restore_best_weights=True, verbose=1),
        # EpochProgressCallback(CFG["epochs"], name),
    ]

    print(f"\n{_c('  ▶ Training:', 'bold', 'cyan')} {_c(name, 'yellow')}")
    t0      = time.time()
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=CFG["epochs"],
        callbacks=callbacks,
        verbose=1,
    )
    elapsed  = time.time() - t0
    best_val = max(history.history["val_accuracy"]) * 100.0
    print(
        f"\n  {_c('✔ Done:', 'green', 'bold')} "
        f"best val acc = {_c(f'{best_val:.2f}%', 'green')}  "
        f"wall time = {_c(f'{elapsed:.0f}s', 'grey')}"
    )
    trained_models[name] = model
    all_histories[name]  = history.history

#  9. FINAL TEST-SET EVALUATION
results = {}
for name, model in trained_models.items():
    test_loss, test_acc_raw = model.evaluate(test_ds_oh, verbose=0)
    test_acc  = test_acc_raw * 100.0
    macro_f1  = compute_macro_f1(model, test_ds)
    results[name] = {
        "test_acc":  round(test_acc, 2),
        "macro_f1":  round(macro_f1, 2),
        "params":    model.count_params(),
        "test_loss": round(float(test_loss), 4),
    }
print_comparison_table(results)

#  10. PERSIST RESULTS
results_path = os.path.join(CFG["results_dir"], "persian_results.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"[INFO] Results   → {results_path}")

histories_path = os.path.join(CFG["results_dir"], "persian_histories.json")
with open(histories_path, "w") as f:
    json.dump(
        {n: {k: [float(v) for v in vals] for k, vals in h.items()}
         for n, h in all_histories.items()},
        f, indent=2,
    )
print(f"[INFO] Histories → {histories_path}")
print(_c("\n[DONE] HODA Persian benchmark complete.\n", "green", "bold"))

In [ ]:
2026-04-25 00:02:47.340541: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
E0000 00:00:1777075367.528692      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777075367.583619      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777075368.026811      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777075368.026881      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777075368.026885      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777075368.026887      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
[INFO] HODA layout detected: CDB mode (HodaDatasetReader)
[INFO] Reading train CDB: /kaggle/input/datasets/invalizare/hoda-mat/DigitDB/Train 60000.cdb
[INFO] Reading test  CDB: /kaggle/input/datasets/invalizare/hoda-mat/DigitDB/Test 20000.cdb
[INFO] Classes in dataset : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[INFO] Train: 54,000 | Val: 6,000 | Test: 20,000
I0000 00:00:1777075401.331013      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1777075401.336978      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5

══════════════════════════════════════════════════════════════════════
  Starting benchmark: 1 model(s) × 100 epochs  [HODA – Persian]
══════════════════════════════════════════════════════════════════════


╔══════════════════════════════════════════════════════════════╗
║  WhatNet-Persian  –  Parameter Summary                       ║
╠══════════════════╦═══════════════════════╦══════════════════╣
║  Layer           ║  Type                 ║           Params  ║
╠══════════════════╬═══════════════════════╬══════════════════╣
║  conv2d          ║  Conv2D               ║              288  ║
║  conv2d_1        ║  Conv2D               ║              288  ║
║  batch_normaliz  ║  BatchNormalization   ║              128  ║
║  batch_normaliz  ║  BatchNormalization   ║              128  ║
║  dense           ║  Dense                ║              520  ║
║  dense_1         ║  Dense                ║              576  ║
║  conv2d_2        ║  Conv2D               ║            4,096  ║
║  batch_normaliz  ║  BatchNormalization   ║              256  ║
║  conv2d_3        ║  Conv2D               ║           36,864  ║
║  batch_normaliz  ║  BatchNormalization   ║              256  ║
║  conv2d_4        ║  Conv2D               ║           36,864  ║
║  batch_normaliz  ║  BatchNormalization   ║              256  ║
║  conv2d_5        ║  Conv2D               ║           36,864  ║
║  batch_normaliz  ║  BatchNormalization   ║              256  ║
║  conv2d_6        ║  Conv2D               ║           36,864  ║
║  batch_normaliz  ║  BatchNormalization   ║              256  ║
║  conv2d_7        ║  Conv2D               ║           36,864  ║
║  batch_normaliz  ║  BatchNormalization   ║              256  ║
║  conv2d_8        ║  Conv2D               ║           36,864  ║
║  batch_normaliz  ║  BatchNormalization   ║              256  ║
║  … (truncated)   ║                       ║                   ║
╠══════════════════╩═══════════════════════╩══════════════════╣
║  Trainable params                      :          5,344,324  ║
║  Non-trainable params                  :              8,212  ║
║  Total params                          :          5,352,536  ║
╚══════════════════════════════════════════════════════════════╝

  ▶ Training: WhatNet-Persian
Epoch 1/100
WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
I0000 00:00:1777075423.788033     124 service.cc:152] XLA service 0x79e478003040 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777075423.788073     124 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1777075423.788079     124 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1777075427.580013     124 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1777075446.568148     124 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
844/844 ━━━━━━━━━━━━━━━━━━━━ 139s 115ms/step - accuracy: 0.8980 - loss: 0.7513 - val_accuracy: 0.9882 - val_loss: 0.5371
Epoch 2/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 75s 88ms/step - accuracy: 0.9928 - loss: 0.5223 - val_accuracy: 0.8283 - val_loss: 1.0354
Epoch 3/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 78s 92ms/step - accuracy: 0.9949 - loss: 0.5176 - val_accuracy: 0.9953 - val_loss: 0.5164
Epoch 4/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 78s 91ms/step - accuracy: 0.9961 - loss: 0.5123 - val_accuracy: 0.9940 - val_loss: 0.5166
Epoch 5/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 77s 91ms/step - accuracy: 0.9963 - loss: 0.5117 - val_accuracy: 0.9920 - val_loss: 0.5199
Epoch 6/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 78s 92ms/step - accuracy: 0.9962 - loss: 0.5118 - val_accuracy: 0.9967 - val_loss: 0.5100
Epoch 7/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 78s 92ms/step - accuracy: 0.9971 - loss: 0.5092 - val_accuracy: 0.9950 - val_loss: 0.5142
Epoch 8/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 78s 91ms/step - accuracy: 0.9966 - loss: 0.5102 - val_accuracy: 0.9918 - val_loss: 0.5192
Epoch 9/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 78s 91ms/step - accuracy: 0.9965 - loss: 0.5094 - val_accuracy: 0.9912 - val_loss: 0.5270
Epoch 10/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 78s 92ms/step - accuracy: 0.9970 - loss: 0.5084 - val_accuracy: 0.9958 - val_loss: 0.5112
Epoch 11/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 78s 92ms/step - accuracy: 0.9977 - loss: 0.5066 - val_accuracy: 0.9973 - val_loss: 0.5068
Epoch 12/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 77s 91ms/step - accuracy: 0.9979 - loss: 0.5067 - val_accuracy: 0.9965 - val_loss: 0.5087
Epoch 13/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 78s 92ms/step - accuracy: 0.9983 - loss: 0.5058 - val_accuracy: 0.9982 - val_loss: 0.5052
Epoch 14/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 78s 91ms/step - accuracy: 0.9980 - loss: 0.5056 - val_accuracy: 0.9980 - val_loss: 0.5061
Epoch 15/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 77s 91ms/step - accuracy: 0.9982 - loss: 0.5053 - val_accuracy: 0.9978 - val_loss: 0.5063
Epoch 16/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 77s 91ms/step - accuracy: 0.9986 - loss: 0.5043 - val_accuracy: 0.9972 - val_loss: 0.5076
Epoch 17/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 77s 91ms/step - accuracy: 0.9986 - loss: 0.5041 - val_accuracy: 0.9947 - val_loss: 0.5130
Epoch 18/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 78s 91ms/step - accuracy: 0.9993 - loss: 0.5030 - val_accuracy: 0.9962 - val_loss: 0.5099
Epoch 19/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 78s 92ms/step - accuracy: 0.9986 - loss: 0.5043 - val_accuracy: 0.9958 - val_loss: 0.5122
Epoch 20/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 78s 92ms/step - accuracy: 0.9980 - loss: 0.5050 - val_accuracy: 0.9982 - val_loss: 0.5045
Epoch 21/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 78s 91ms/step - accuracy: 0.9985 - loss: 0.5041 - val_accuracy: 0.9972 - val_loss: 0.5082
Epoch 22/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 78s 91ms/step - accuracy: 0.9993 - loss: 0.5026 - val_accuracy: 0.9972 - val_loss: 0.5071
Epoch 23/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 78s 92ms/step - accuracy: 0.9994 - loss: 0.5024 - val_accuracy: 0.9983 - val_loss: 0.5053
Epoch 24/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 78s 92ms/step - accuracy: 0.9995 - loss: 0.5023 - val_accuracy: 0.9978 - val_loss: 0.5061
Epoch 25/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 79s 93ms/step - accuracy: 0.9991 - loss: 0.5028 - val_accuracy: 0.9988 - val_loss: 0.5041
Epoch 26/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 78s 91ms/step - accuracy: 0.9994 - loss: 0.5021 - val_accuracy: 0.9967 - val_loss: 0.5082
Epoch 27/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 78s 91ms/step - accuracy: 0.9994 - loss: 0.5023 - val_accuracy: 0.9982 - val_loss: 0.5058
Epoch 28/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 78s 92ms/step - accuracy: 0.9995 - loss: 0.5019 - val_accuracy: 0.9988 - val_loss: 0.5039
Epoch 29/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 77s 91ms/step - accuracy: 0.9994 - loss: 0.5020 - val_accuracy: 0.9982 - val_loss: 0.5050
Epoch 30/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 78s 91ms/step - accuracy: 0.9993 - loss: 0.5023 - val_accuracy: 0.9973 - val_loss: 0.5076
Epoch 31/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 78s 92ms/step - accuracy: 0.9992 - loss: 0.5023 - val_accuracy: 0.9977 - val_loss: 0.5065
Epoch 32/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 78s 91ms/step - accuracy: 0.9990 - loss: 0.5025 - val_accuracy: 0.9977 - val_loss: 0.5064
Epoch 33/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 77s 91ms/step - accuracy: 0.9994 - loss: 0.5020 - val_accuracy: 0.9980 - val_loss: 0.5052
Epoch 34/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 78s 92ms/step - accuracy: 0.9997 - loss: 0.5015 - val_accuracy: 0.9990 - val_loss: 0.5030
Epoch 35/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 78s 91ms/step - accuracy: 0.9996 - loss: 0.5016 - val_accuracy: 0.9980 - val_loss: 0.5048
Epoch 36/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 78s 91ms/step - accuracy: 0.9994 - loss: 0.5023 - val_accuracy: 0.9987 - val_loss: 0.5038
Epoch 37/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 78s 91ms/step - accuracy: 0.9994 - loss: 0.5019 - val_accuracy: 0.9983 - val_loss: 0.5052
Epoch 38/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 78s 91ms/step - accuracy: 0.9997 - loss: 0.5013 - val_accuracy: 0.9985 - val_loss: 0.5038
Epoch 39/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 78s 91ms/step - accuracy: 0.9997 - loss: 0.5012 - val_accuracy: 0.9977 - val_loss: 0.5060
Epoch 40/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 78s 91ms/step - accuracy: 0.9997 - loss: 0.5013 - val_accuracy: 0.9985 - val_loss: 0.5041
Epoch 41/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 77s 91ms/step - accuracy: 0.9997 - loss: 0.5012 - val_accuracy: 0.9980 - val_loss: 0.5057
Epoch 42/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 77s 91ms/step - accuracy: 0.9997 - loss: 0.5013 - val_accuracy: 0.9977 - val_loss: 0.5057
Epoch 43/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 78s 92ms/step - accuracy: 0.9998 - loss: 0.5011 - val_accuracy: 0.9980 - val_loss: 0.5053
Epoch 44/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 78s 92ms/step - accuracy: 0.9997 - loss: 0.5012 - val_accuracy: 0.9983 - val_loss: 0.5043
Epoch 45/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 78s 92ms/step - accuracy: 0.9998 - loss: 0.5008 - val_accuracy: 0.9992 - val_loss: 0.5028
Epoch 46/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 78s 92ms/step - accuracy: 0.9996 - loss: 0.5013 - val_accuracy: 0.9987 - val_loss: 0.5038
Epoch 47/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 77s 91ms/step - accuracy: 0.9998 - loss: 0.5010 - val_accuracy: 0.9988 - val_loss: 0.5032
Epoch 48/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 77s 91ms/step - accuracy: 0.9998 - loss: 0.5009 - val_accuracy: 0.9985 - val_loss: 0.5035
Epoch 49/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 78s 91ms/step - accuracy: 0.9998 - loss: 0.5008 - val_accuracy: 0.9988 - val_loss: 0.5037
Epoch 50/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 77s 91ms/step - accuracy: 0.9999 - loss: 0.5007 - val_accuracy: 0.9988 - val_loss: 0.5029
Epoch 51/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 78s 91ms/step - accuracy: 0.9999 - loss: 0.5006 - val_accuracy: 0.9985 - val_loss: 0.5046
Epoch 52/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 78s 91ms/step - accuracy: 0.9998 - loss: 0.5010 - val_accuracy: 0.9985 - val_loss: 0.5043
Epoch 53/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 78s 91ms/step - accuracy: 0.9999 - loss: 0.5006 - val_accuracy: 0.9982 - val_loss: 0.5055
Epoch 54/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 77s 91ms/step - accuracy: 0.9998 - loss: 0.5007 - val_accuracy: 0.9978 - val_loss: 0.5058
Epoch 55/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 77s 91ms/step - accuracy: 0.9997 - loss: 0.5009 - val_accuracy: 0.9985 - val_loss: 0.5048
Epoch 56/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 77s 91ms/step - accuracy: 0.9999 - loss: 0.5008 - val_accuracy: 0.9988 - val_loss: 0.5032
Epoch 57/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 77s 91ms/step - accuracy: 1.0000 - loss: 0.5005 - val_accuracy: 0.9985 - val_loss: 0.5036
Epoch 58/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 77s 91ms/step - accuracy: 0.9999 - loss: 0.5006 - val_accuracy: 0.9990 - val_loss: 0.5025
Epoch 59/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 77s 91ms/step - accuracy: 0.9998 - loss: 0.5007 - val_accuracy: 0.9985 - val_loss: 0.5044
Epoch 60/100
844/844 ━━━━━━━━━━━━━━━━━━━━ 78s 91ms/step - accuracy: 0.9999 - loss: 0.5006 - val_accuracy: 0.9987 - val_loss: 0.5043
Epoch 60: early stopping
Restoring model weights from the end of the best epoch: 45.

  ✔ Done: best val acc = 99.92%  wall time = 4717s

╔══════════════════════════════════════════════════════════════════════╗
║  FINAL TEST-SET COMPARISON                                           ║
╠════════════════════════╦════════════╦════════════╦════════════╦══════╣
║  Model                 ║     Params ║   Test Acc ║   Macro F1 ║ Loss ║
╠════════════════════════╬════════════╬════════════╬════════════╬══════╣
║★ WhatNet-Persian       ║ 5,344,324 ║     99.72%║     99.72%║0.508 ║
╚════════════════════════╩════════════╩════════════╩════════════╩══════╝

  ★  Winner: WhatNet-Persian  (99.72% test accuracy)

[INFO] Results   → ./results/persian_results.json
[INFO] Histories → ./results/persian_histories.json

[DONE] HODA Persian benchmark complete.